# Cartographie des Unités Géologiques avec Prithvi EO

Ce notebook segmente le terrain en unités géologiques cohérentes en utilisant le clustering hiérarchique sur les caractéristiques profondes de l'IA.

In [ ]:
!pip install geemap earthengine-api scikit-learn rasterio terratorch torch matplotlib seaborn -q
import ee, geemap, torch, rasterio
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.spatial.distance import pdist
from terratorch import BACKBONE_REGISTRY

ee.Initialize(project='geocongoai-api')

In [ ]:
roi = ee.Geometry.Rectangle([15.0, -5.0, 15.5, -4.5])
image = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED").filterBounds(roi).median().clip(roi)
geemap.ee_export_image(image.select(['B2', 'B3', 'B4', 'B8', 'B11', 'B12']), 'input.tif', scale=30, region=roi)

In [ ]:
model = BACKBONE_REGISTRY.build("prithvi_eo_v2_300", num_frames=1, in_chans=6, pretrained=True).eval().to('cpu')
with rasterio.open('input.tif') as src: img = src.read().astype(np.float32) / 10000.0
with torch.no_grad():
    out = model(torch.from_numpy(img).unsqueeze(0))
    feats = out[0] if isinstance(out, list) else out
    
feats_np = feats[0, 1:].numpy()
h_feat = int(np.sqrt(feats_np.shape[0]))

# Clustering Hiérarchique (Ward + Cosine)
dist_matrix = pdist(feats_np, metric='cosine')
linkage_matrix = linkage(dist_matrix, method='ward')
units = fcluster(linkage_matrix, 8, criterion='maxclust')
unit_map = units.reshape(h_feat, -1)

plt.imshow(unit_map, cmap='terrain')
plt.colorbar(label='Unités Géologiques')
plt.title("Segmentation des Unités Géologiques (Ward Clustering)")
plt.show()